# Brecha educativa urbano vs rural — exportación de gráficos

Ejecuta esta celda completa en Google Colab. Montará Google Drive, procesará las bases ENAHO y exportará los diez gráficos Plotly originales a `/content/graficos_brecha_educativa/`.

In [ ]:
# -*- coding: utf-8 -*-
"""Brecha educativa urbano vs rural

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1I7qX3rJ79-0z04HweJQpUnL62hBMk513
"""

#@title INTRODUCCIÓN Y OBJETIVO – Brecha educativa urbano vs rural { display-mode: "form" }
# Presentación del proyecto: introducción, objetivos, datos usados y cómo leer el notebook.
from IPython.display import HTML, display
display(HTML('''
<div style="font-family:Segoe UI,Roboto,Arial,sans-serif;max-width:1000px;color:#1f2d3d">
  <div style="background:linear-gradient(135deg,#1f3a5f,#2a78d6);color:#fff;border-radius:14px 14px 0 0;padding:26px 28px">
    <div style="font-size:13px;letter-spacing:1px;font-weight:700;opacity:.85">ANÁLISIS DE DATOS · ENAHO – INEI</div>
    <div style="font-size:30px;font-weight:800;margin:6px 0 4px">Brecha educativa urbano vs rural en el Perú</div>
    <div style="font-size:17px;opacity:.95">2004 – 2025 · ¿Cuánto cambió la distancia entre la ciudad y el campo?</div>
  </div>
  <div style="background:#f7f9fc;border:1px solid #e3e9f2;border-top:none;border-radius:0 0 14px 14px;padding:22px 28px;box-shadow:0 2px 10px rgba(0,0,0,.08)">

    <div style="font-size:19px;font-weight:700;margin-bottom:6px">📖 Introducción</div>
    <div style="font-size:15px;line-height:1.6">
      En el Perú, el lugar donde naces influye en hasta dónde llegas a estudiar. En las últimas dos décadas el país amplió el acceso a la escuela,
      pero la pregunta es si ese avance llegó por igual a la ciudad y al campo. Este notebook usa la <b>Encuesta Nacional de Hogares (ENAHO)</b> del INEI
      para comparar el nivel educativo del área <b style="color:#2a78d6">urbana</b> con el del área <b style="color:#eb6834">rural</b> a lo largo de 21 años,
      y luego prueba un modelo predictivo para ver qué características pesan más.
    </div>

    <div style="font-size:19px;font-weight:700;margin:20px 0 6px">🎯 Objetivo general</div>
    <div style="font-size:15px;line-height:1.6;background:#fff;border-left:6px solid #2a78d6;border-radius:8px;padding:12px 16px">
      Medir cómo cambió la brecha educativa entre el área urbana y la rural del Perú entre 2004 y 2025, e identificar dónde y en quiénes se concentra.
    </div>

    <div style="font-size:19px;font-weight:700;margin:20px 0 6px">✅ Objetivos específicos</div>
    <div style="font-size:15px;font-weight:700;margin:4px 0 2px;color:#2a78d6">Parte 1 · Análisis descriptivo (gráficos 1 a 8)</div>
    <ol style="font-size:15px;line-height:1.7;margin:0 0 8px 20px;padding:0">
      <li><b>Evolución:</b> ver cómo mejoran la ciudad y el campo en 6 indicadores de educación.</li>
      <li><b>Brecha:</b> medir si la distancia entre ambas áreas se achicó con los años.</li>
      <li><b>Nivel alcanzado:</b> conocer hasta dónde estudian las personas de 25+ años en cada área.</li>
      <li><b>Generaciones:</b> comprobar si los jóvenes rurales llegan más lejos que sus padres y abuelos.</li>
      <li><b>Departamentos:</b> identificar dónde la brecha es mayor y menor.</li>
      <li><b>Área × sexo:</b> ver cómo se combinan la brecha urbano-rural y la de género.</li>
      <li><b>Lengua materna:</b> evaluar si hablar una lengua originaria suma desventaja.</li>
      <li><b>Burbujas por departamento:</b> relacionar ruralidad y educación en cada departamento a lo largo de los años.</li>
    </ol>
    <div style="font-size:15px;font-weight:700;margin:4px 0 2px;color:#eb6834">Parte 2 · Modelo predictivo (árbol de decisión y gráficos 9 y 10)</div>
    <ol start="9" style="font-size:15px;line-height:1.7;margin:0 0 0 20px;padding:0">
      <li><b>Árbol de decisión:</b> probar si el contexto de una persona (área, sexo, lengua, departamento y edad) permite predecir si terminó la secundaria.</li>
      <li><b>Matriz de confusión:</b> ver en qué casos acierta y en cuáles se equivoca el modelo.</li>
      <li><b>Importancia de variables:</b> identificar qué características pesan más en la predicción.</li>
    </ol>

    <div style="font-size:19px;font-weight:700;margin:20px 0 6px">🗂️ Datos y definiciones</div>
    <table style="border-collapse:collapse;font-size:14px;width:100%;background:#fff;border-radius:8px;overflow:hidden">
      <tr><td style="padding:9px 14px;font-weight:700;width:190px;background:#eef3fa">Fuente</td><td style="padding:9px 14px">INEI – ENAHO, módulo 300 (Educación). Más de 2.2 millones de registros.</td></tr>
      <tr><td style="padding:9px 14px;font-weight:700;background:#eef3fa">Periodo</td><td style="padding:9px 14px">2004–2025. Se excluye 2026 porque solo tiene dos trimestres y su ponderación no es comparable.</td></tr>
      <tr><td style="padding:9px 14px;font-weight:700;background:#eef3fa">Urbano / Rural</td><td style="padding:9px 14px">Urbano: centros poblados de 2,000 habitantes o más. Rural: menos de 2,000.</td></tr>
      <tr><td style="padding:9px 14px;font-weight:700;background:#eef3fa">Ponderación</td><td style="padding:9px 14px">Todas las cifras usan el factor de expansión de la ENAHO (resultados representativos del país).</td></tr>
      <tr><td style="padding:9px 14px;font-weight:700;background:#eef3fa">Brecha</td><td style="padding:9px 14px">Valor urbano menos valor rural. En puntos porcentuales (pp), o en años para años de estudio.</td></tr>
      <tr><td style="padding:9px 14px;font-weight:700;background:#eef3fa">Modelo</td><td style="padding:9px 14px">Árbol de decisión (profundidad 6) con datos 2023–2025 de personas de 25+ años: 70% para entrenar y 30% para probar.</td></tr>
      <tr><td style="padding:9px 14px;font-weight:700;background:#eef3fa">Ojo con 2020</td><td style="padding:9px 14px">Ese año la encuesta fue telefónica por la COVID-19; los cambios de 2020 se deben leer con cautela.</td></tr>
    </table>

    <div style="font-size:14px;color:#44546a;margin-top:16px">
      👉 <b>Cómo usar este notebook:</b> ejecuta las celdas en orden. Cada gráfico (y el modelo) tiene una caja azul con su objetivo antes y una caja naranja con su análisis después. Los gráficos son interactivos (menús, controles de año y valores al pasar el cursor). Al final está el resumen de hallazgos.
    </div>
  </div>
</div>'''))

# CELDA 1 – CONFIGURACIÓN
# Instala Plotly, conecta Google Drive, importa las librerías y busca
# automáticamente la carpeta DataSet (ENAHO 2004–2025) para definir BASE.
!pip -q install plotly
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go

# EXPORTACIÓN DE LOS GRÁFICOS ORIGINALES (sin alterar sus datos, trazas ni diseño)
OUT_CHARTS = '/content/graficos_brecha_educativa'
os.makedirs(OUT_CHARTS, exist_ok=True)


BASE = None
for root, dirs, files in os.walk('/content/drive/MyDrive', followlinks=True):
    if 'enaho01a-2004-300.dta' in files:
        BASE = os.path.dirname(root); break
print('BASE =', BASE)
if BASE is None:
    print('No la encontré. Esto ve Colab en Mi unidad:', os.listdir('/content/drive/MyDrive'))
    raise SystemExit

# CELDA 2 – CARGA DE DATOS
# Lee el módulo 300 (Educación) de cada año 2004–2025, conserva solo las variables
# necesarias y las une en una sola tabla de personas (P). Crea variables derivadas:
# área (urbano/rural), sexo, lengua materna y años de estudio aproximados.
# 2026 queda fuera porque su ponderación no es comparable con la serie.
COLS = {'P301A':'nivel','P301B':'anio_est','P208A':'edad','P207':'sexo','P302':'sabe_leer',
        'P306':'matric','P300A':'lengua','ESTRATO':'estrato','FACTOR07':'peso'}
partes = []
for y in range(2004, 2026):                      # 2026 fuera: su ponderación no es comparable
    p = f'{BASE}/{y}/enaho01a-{y}-300.dta'       # nombre exacto (evita el 300a y el archivo 2009 de la carpeta 2010)
    d = pd.read_stata(p, convert_categoricals=False, convert_dates=False)
    d.columns = [c.upper() for c in d.columns]
    t = d[list(COLS)].apply(pd.to_numeric, errors='coerce').rename(columns=COLS)
    t['depto'] = d['UBIGEO'].astype(str).str.zfill(6).str[:2].astype(int)
    t['anio'] = y
    partes.append(t); print(y, 'ok', len(t)); del d
P = pd.concat(partes, ignore_index=True); del partes

P['area'] = np.where(P.estrato <= 5, 'Urbano', 'Rural')            # urbano = estratos 1-5 (2 000+ hab.)
P['sexo'] = P.sexo.map({1:'Hombre', 2:'Mujer'})
P['lengua_g'] = np.select([P.lengua == 4, P.lengua.isin([1,2,3]) | (P.lengua >= 10)], ['Castellano','Lengua originaria'], 'Otra')
P = P[P.nivel.notna() & (P.nivel != 99) & P.estrato.notna()].copy()
P['nivel'] = P.nivel.astype(int)

# años de estudio aproximados (supuesto: sup. no univ. completa=14, univ. completa=16, maestría=18, doctorado=21)
n, a = P.nivel.values, P.anio_est.fillna(-1).values
P['anios_est'] = np.select(
    [np.isin(n,[1,2]), n==3, n==4, n==5, n==6, n==7, n==8, n==9, n==10, np.isin(n,[11,13]), n==14],
    [0, np.where(a>=0, np.minimum(a,5), 3), 6, np.where(a>=0, 6+np.minimum(a,5), 8), 11,
     np.where(a>=0, 11+np.minimum(a,3), 12), 14, np.where(a>=0, 11+np.minimum(a,5), 13), 16, 18, 21], np.nan)
print(P.shape); P.head()

# CELDA 3 – INDICADORES
# Calcula por año y área (urbano/rural), ponderando con el factor de expansión:
# secundaria completa o más, estudios universitarios, sin nivel o primaria incompleta,
# años promedio de estudio (25+), analfabetismo (15+) y matrícula de 12–16 años.
# Guarda la tabla en CSV y muestra la comparación 2004 vs 2025.
NIV = {1:'Sin nivel/inicial', 2:'Sin nivel/inicial', 3:'Primaria', 4:'Primaria', 5:'Secundaria incompleta', 6:'Secundaria completa',
       7:'Superior no univ.', 8:'Superior no univ.', 9:'Superior univ.', 10:'Superior univ.', 11:'Maestría/doctorado', 13:'Maestría/doctorado', 14:'Maestría/doctorado'}
SEC, UNI, BAJ = [6,7,8,9,10,11,13,14], [9,10,11,13,14], [1,3]
wm = lambda x, w, m=None: np.nan if w.sum() == 0 else (x*w).sum()/w.sum()*100

def indicadores(df):
    filas = []
    for (y, ar), g in df.groupby(['anio','area']):
        a25 = g[g.edad >= 25]; a15 = g[g.edad >= 15]; e = g[(g.edad >= 12) & (g.edad <= 16)]
        w = a25.peso
        filas += [
         (y,ar,'Secundaria completa o más (25+)',   wm(a25.nivel.isin(SEC), w)),
         (y,ar,'Estudios universitarios (25+)',     wm(a25.nivel.isin(UNI), w)),
         (y,ar,'Sin nivel o primaria incompleta (25+)', wm(a25.nivel.isin(BAJ), w)),
         (y,ar,'Años promedio de estudio (25+)',    (a25.anios_est*w).sum()/w[a25.anios_est.notna()].sum()),
         (y,ar,'Analfabetismo (15+)',               wm(a15.sabe_leer.eq(2), a15.peso)),
         (y,ar,'Matriculados 12–16 años',           wm(e.matric.eq(1), e.peso)),
        ]
    return pd.DataFrame(filas, columns=['anio','area','indicador','valor'])

IND = indicadores(P)
IND_W = IND.pivot_table(index=['indicador','anio'], columns='area', values='valor').reset_index()
IND_W['brecha'] = IND_W.Urbano - IND_W.Rural
IND.to_csv('indicadores_educacion_urbano_rural.csv', index=False)
display(IND_W[IND_W.anio.isin([2004, 2025])].round(1))

#@title CELDA 3B – Estilo de las cajas de título y análisis (ejecútala una vez) { display-mode: "form" }
# Define dos funciones: intro() = caja azul con título y objetivo (va ANTES de cada gráfico)
# y analisis() = caja naranja con los hallazgos y una conclusión (va DESPUÉS de cada gráfico).
from IPython.display import HTML, display

def intro(n, titulo, objetivo, leer=''):
    badge = n if isinstance(n, str) else f'GRÁFICO {n}'   # n puede ser un número o un texto (ej. 'MODELO')
    tip = (f'<div style="margin-top:10px;font-size:14px;color:#44546a"><b>👀 Cómo leerlo:</b> {leer}</div>') if leer else ''
    display(HTML(f'''
    <div style="font-family:Segoe UI,Roboto,Arial,sans-serif;max-width:1000px;background:#f2f7ff;border-left:7px solid #2a78d6;
                border-radius:10px;padding:16px 20px;margin:14px 0 4px;box-shadow:0 2px 8px rgba(0,0,0,.08);color:#1f2d3d">
      <span style="background:#1f3a5f;color:#fff;font-size:12px;font-weight:700;letter-spacing:.5px;padding:4px 10px;border-radius:12px">{badge}</span>
      <div style="font-size:21px;font-weight:700;margin:8px 0 6px">{titulo}</div>
      <div style="font-size:15px"><b>🎯 Objetivo:</b> {objetivo}</div>{tip}
    </div>'''))

def analisis(puntos, conclusion):
    lis = ''.join(f'<li style="margin:6px 0">{p}</li>' for p in puntos)
    display(HTML(f'''
    <div style="font-family:Segoe UI,Roboto,Arial,sans-serif;max-width:1000px;background:#fff7f1;border-left:7px solid #eb6834;
                border-radius:10px;padding:16px 20px;margin:4px 0 22px;box-shadow:0 2px 8px rgba(0,0,0,.08);color:#1f2d3d">
      <div style="font-size:18px;font-weight:700;margin-bottom:4px">📌 Análisis</div>
      <ul style="margin:6px 0 12px 18px;padding:0;font-size:15px;line-height:1.5">{lis}</ul>
      <div style="background:#1f3a5f;color:#fff;border-radius:8px;padding:10px 14px;font-size:15px"><b>💡 Conclusión:</b> {conclusion}</div>
    </div>'''))

#@title Intro gráfico 1 { display-mode: "form" }
intro(1, 'Evolución urbano vs rural, 2004–2025',
      'Ver si el área urbana y la rural mejoran en educación y a qué ritmo lo hacen.',
      'Usa el menú desplegable para cambiar de indicador. La línea punteada marca 2020 (encuesta telefónica por COVID).')

# CELDA 4 – GRÁFICO 1: EVOLUCIÓN URBANO VS RURAL
# Líneas 2004-2025 con menú desplegable para cambiar de indicador.
# Cada punto muestra su valor (% o años). El título explica qué mide cada indicador. La línea punteada marca 2020 (encuesta telefónica por COVID).
AZUL, NAR = '#2a78d6', '#eb6834'
INDS = list(IND.indicador.unique())

# Título (pregunta) y descripción de cada indicador, en lenguaje simple
TIT = {
 'Secundaria completa o más (25+)':        ('¿Cuántas personas terminaron la secundaria?', '% de personas de 25 años o más con secundaria completa o más'),
 'Estudios universitarios (25+)':          ('¿Cuántas personas llegaron a la universidad?', '% de personas de 25 años o más con estudios universitarios (completos o incompletos) o posgrado'),
 'Sin nivel o primaria incompleta (25+)':  ('¿Cuántas personas no terminaron ni la primaria?', '% de personas de 25 años o más sin ningún nivel o con primaria incompleta'),
 'Años promedio de estudio (25+)':         ('¿Cuántos años estudian en promedio?', 'Años promedio de estudio de las personas de 25 años o más (aproximado)'),
 'Analfabetismo (15+)':                    ('¿Cuántas personas no saben leer ni escribir?', '% de personas de 15 años o más que no sabe leer ni escribir'),
 'Matriculados 12–16 años':                ('¿Cuántos adolescentes están matriculados?', '% de adolescentes de 12 a 16 años matriculados en el sistema educativo'),
}
def titulo(ind): return f'<b>{TIT[ind][0]}</b><br><sup>{TIT[ind][1]} · Urbano vs rural, 2004–2025</sup>'
def ytit(ind):   return 'Años de estudio' if 'Años' in ind else '% de personas'

fig = go.Figure()
for i, ind in enumerate(INDS):
    suf = '' if 'Años' in ind else '%'                      # unidad que se escribe junto al número
    for ar, col in [('Urbano', AZUL), ('Rural', NAR)]:
        d = IND[(IND.indicador == ind) & (IND.area == ar)]
        fig.add_trace(go.Scatter(x=d.anio, y=d.valor, name=ar, mode='lines+markers+text', line=dict(color=col, width=4), marker=dict(size=7),
                                 text=[f'{x:.1f}{suf}' for x in d.valor], textposition='top center' if ar == 'Urbano' else 'bottom center',
                                 textfont=dict(size=10, color=col), cliponaxis=False, visible=(i == 0),
                                 hovertemplate='%{x}: <b>%{y:.1f}'+suf+'</b><extra>'+ar+'</extra>'))
botones = [dict(label=ind, method='update', args=[{'visible': [j//2 == i for j in range(2*len(INDS))]}, {'title.text': titulo(ind), 'yaxis.title.text': ytit(ind)}]) for i, ind in enumerate(INDS)]
fig.update_layout(title=dict(text=titulo(INDS[0]), x=0, xanchor='left', y=0.97, yanchor='top'),
                  template='plotly_white', height=660, hovermode='x unified', margin=dict(t=170),
                  yaxis_title=ytit(INDS[0]), yaxis=dict(rangemode='tozero'),
                  updatemenus=[dict(buttons=botones, direction='down', x=0, y=1.10, xanchor='left', yanchor='bottom')],
                  xaxis=dict(dtick=2), legend=dict(orientation='h', y=-0.12),
                  annotations=[dict(x=2020, y=1.02, yref='paper', text='2020: encuesta telefónica (COVID)', showarrow=False, font=dict(color='gray', size=11))],
                  shapes=[dict(type='line', x0=2020, x1=2020, y0=0, y1=1, yref='paper', line=dict(color='gray', dash='dash', width=1))])
fig.write_html(os.path.join(OUT_CHARTS, 'grafico_01.html'), include_plotlyjs='cdn', full_html=True)

#@title Análisis gráfico 1 { display-mode: "form" }
analisis([
  'Secundaria completa o más (25+): urbano <b>57.0% → 66.2%</b>; rural <b>14.2% → 24.9%</b>. El rural creció más rápido (+10.7 pp vs +9.2 pp), pero partía de muy abajo.',
  'Sin nivel o primaria incompleta: en el área rural bajó de <b>56.2% a 42.6%</b>; en la urbana, de 17.5% a 13.6%.',
  'Analfabetismo (15+): rural <b>25.4% → 17.7%</b>; urbano 5.9% → 4.1%.',
  'Años promedio de estudio (25+): rural <b>4.3 → 5.4</b>; urbano 9.5 → 10.2.',
  'Matrícula de 12–16 años: rural <b>63.7% → 85.4%</b>; urbano 76.2% → 90.1%.',
  'El salto rural de 2020 coincide con el cambio a encuesta telefónica: léelo con cautela.'],
  'Todo mejora en ambas áreas y el campo avanza más rápido, pero el rural de 2025 (24.9%) sigue lejos del urbano de 2004 (57.0%).')

#@title Intro gráfico 2 { display-mode: "form" }
intro(2, '¿La brecha se cerró? Diferencia urbano − rural',
      'Medir cuánto se achicó (o no) la distancia entre el área urbana y la rural año a año.',
      'Cada barra NO es una de las áreas: es el valor urbano menos el valor rural (cuántos puntos más tiene la ciudad que el campo). Si las barras bajan con los años, la brecha se cierra. Pasa el cursor para ver urbano, rural y brecha.')

# CELDA 5 – GRÁFICO 2: ¿LA BRECHA SE CERRÓ? (diferencia urbano – rural)
# Cada barra = valor urbano MENOS valor rural (no es una de las dos áreas). Menú desplegable para cambiar de indicador.
# Al pasar el cursor se ven los tres valores: urbano, rural y la brecha. Si las barras bajan con los años, la brecha se cierra.
MORADO = '#6b4fb3'   # color propio de la brecha, para no confundirlo con azul = urbano ni naranja = rural
def titulo2(ind):
    u = 'años' if 'Años' in ind else 'puntos porcentuales (pp)'
    neg = ' Barra negativa = el campo tiene más que la ciudad (aquí lo "malo" es más alto en el campo).' if ind.startswith(('Analfabetismo', 'Sin nivel')) else ''
    return f'<b>Brecha urbano − rural: {TIT[ind][0]}</b><br><sup>Cada barra = valor urbano menos valor rural, en {u}. Si las barras bajan, la brecha se achica.{neg}</sup>'

fig = go.Figure()
for i, ind in enumerate(INDS):
    d = IND_W[IND_W.indicador == ind]
    suf = ' años' if 'Años' in ind else ' pp'                # unidad de la brecha
    uni = '' if 'Años' in ind else '%'                       # unidad de cada área
    fig.add_trace(go.Bar(x=d.anio, y=d.brecha, marker_color=MORADO, visible=(i == 0), name='Brecha',
                         customdata=d[['Urbano', 'Rural']].values,
                         text=[f'{x:.1f}' for x in d.brecha], textposition='outside', textfont=dict(size=11), cliponaxis=False,
                         hovertemplate='<b>%{x}</b><br>Urbano: %{customdata[0]:.1f}'+uni+'<br>Rural: %{customdata[1]:.1f}'+uni+'<br><b>Brecha: %{y:.1f}'+suf+'</b><extra></extra>'))
botones = [dict(label=ind, method='update', args=[{'visible': [j == i for j in range(len(INDS))]}, {'title.text': titulo2(ind), 'yaxis.title.text': ('Brecha (años)' if 'Años' in ind else 'Brecha (pp)')}]) for i, ind in enumerate(INDS)]
fig.update_layout(title=dict(text=titulo2(INDS[0]), x=0, xanchor='left', y=0.97, yanchor='top'),
                  template='plotly_white', height=640, showlegend=False, margin=dict(t=170),
                  yaxis_title='Brecha (pp)',
                  updatemenus=[dict(buttons=botones, direction='down', x=0, y=1.10, xanchor='left', yanchor='bottom')],
                  xaxis=dict(dtick=2))
fig.write_html(os.path.join(OUT_CHARTS, 'grafico_02.html'), include_plotlyjs='cdn', full_html=True)

#@title Análisis gráfico 2 { display-mode: "form" }
analisis([
  'Secundaria completa o más: la brecha pasó de <b>42.8 a 41.3 pp</b>. Casi no se movió en 21 años (−1.5 pp).',
  'Estudios universitarios: la brecha <b>se amplió</b> de 14.5 a 16.1 pp. Es el único indicador que empeora.',
  'Sin nivel o primaria incompleta: la brecha se achicó de 38.7 a <b>29.0 pp</b> (−9.7 pp).',
  'Analfabetismo: de 19.5 a <b>13.6 pp</b> (−5.9 pp).',
  'Matrícula 12–16 años: de 12.5 a <b>4.7 pp</b>, la mayor reducción relativa.'],
  'La brecha se cierra rápido en acceso (matrícula, alfabetización), pero casi no en logros altos como terminar secundaria o ir a la universidad.')

#@title Intro gráfico 3 { display-mode: "form" }
intro(3, '¿Hasta dónde estudian? Nivel educativo alcanzado',
      'Ver cómo se reparte el nivel educativo de las personas de 25+ años en cada área.',
      'Mueve el control de año o pulsa ▶ para ver la evolución. Cada barra es el % de personas de 25+ en ese nivel.')

# CELDA 6 – GRÁFICO 3: NIVEL EDUCATIVO ALCANZADO (ANIMADO)
# Distribución de personas de 25+ años por nivel educativo, urbano vs rural.
# Usa el control de año o el botón ▶ para ver cómo cambia entre 2004 y 2025.
A25 = P[P.edad >= 25].copy(); A25['grupo'] = A25.nivel.map(NIV); A25 = A25[A25.grupo.notna()]
orden = ['Sin nivel/inicial','Primaria','Secundaria incompleta','Secundaria completa','Superior no univ.','Superior univ.','Maestría/doctorado']
dist = (A25.groupby(['anio','area','grupo']).peso.sum() / A25.groupby(['anio','area']).peso.sum() * 100).rename('pct').reset_index()
fig = px.bar(dist, x='pct', y='grupo', color='area', barmode='group', orientation='h', animation_frame='anio',
             category_orders={'grupo': orden[::-1], 'area': ['Urbano','Rural']}, color_discrete_map={'Urbano': AZUL, 'Rural': NAR},
             range_x=[0, 55], text=dist.pct.round(1).astype(str) + '%', labels={'pct':'% de personas de 25+ años','grupo':'','area':''},
             title='<b>Nivel educativo alcanzado</b> (25+ años): urbano vs rural', template='plotly_white', height=620)
fig.update_traces(textposition='outside'); fig.write_html(os.path.join(OUT_CHARTS, 'grafico_03.html'), include_plotlyjs='cdn', full_html=True)

#@title Análisis gráfico 3 { display-mode: "form" }
analisis([
  'En el área rural, <b>49.4%</b> de las personas de 25+ solo llegó a primaria y otro <b>15.0%</b> no tiene nivel (suman 64.4%). En el urbano suman 23.4%.',
  'Rural: sin nivel/inicial bajó de 22.8% a 15.0% y secundaria completa subió de 9.0% a <b>17.5%</b> (2004 → 2025).',
  'Urbano: secundaria completa pasó de 26.5% a 31.6% y superior universitaria de 15.0% a 16.7%.',
  'Superior universitaria en el campo apenas pasó de 1.8% a <b>2.5%</b>.'],
  'En el campo la mayoría sigue sin pasar de primaria; el avance real está en pasar de "sin nivel" a primaria y secundaria.')

#@title Intro gráfico 4 { display-mode: "form" }
intro(4, 'La brecha por generaciones',
      'Ver si los jóvenes rurales ya llegan más lejos que sus padres y abuelos.',
      'Mueve el control de año. Cada grupo de barras es un grupo de edad; compara el azul (urbano) con el naranja (rural).')

# CELDA 7 – GRÁFICO 4: BRECHA POR GENERACIONES (ANIMADO)
# % con secundaria completa o más por grupo de edad (25–34 … 65+), urbano vs rural.
# Muestra si los jóvenes rurales ya llegan más lejos que las generaciones mayores.
G = P[P.edad >= 25].copy()
G['gen'] = pd.cut(G.edad, [24,34,44,54,64,120], labels=['25–34','35–44','45–54','55–64','65+'])
gen = G.groupby(['anio','area','gen'], observed=True).apply(lambda g: wm(g.nivel.isin(SEC), g.peso)).rename('pct').reset_index()
fig = px.bar(gen, x='gen', y='pct', color='area', barmode='group', animation_frame='anio', range_y=[0, 100],
             color_discrete_map={'Urbano': AZUL, 'Rural': NAR}, category_orders={'area': ['Urbano','Rural']}, text=gen.pct.round(0).astype(int).astype(str) + '%',
             labels={'pct':'% con secundaria completa o más','gen':'Grupo de edad','area':''}, template='plotly_white', height=560,
             title='<b>Secundaria completa o más por grupo de edad</b>: la brecha por generaciones')
fig.update_traces(textposition='outside'); fig.write_html(os.path.join(OUT_CHARTS, 'grafico_04.html'), include_plotlyjs='cdn', full_html=True)

#@title Análisis gráfico 4 { display-mode: "form" }
analisis([
  'Jóvenes de 25–34 años con secundaria completa o más: rural <b>26.9% → 56.0%</b>; urbano 73.3% → 86.5%. La brecha en jóvenes bajó de 46.4 a <b>30.5 pp</b>.',
  'En 2025, los jóvenes rurales (56.0%) ya alcanzan lo que tenían los urbanos de 45–54 años en 2004 (55.4%).',
  'Entre mayores de 65 años en el campo, solo <b>7.6%</b> terminó secundaria (2025). En 25–34 años es 56.0%.',
  'Cada año compara personas distintas por grupo de edad, no a las mismas personas en el tiempo.'],
  'La brecha se cierra por relevo generacional: cada generación rural llega más lejos que la anterior.')

#@title Intro gráfico 5 { display-mode: "form" }
intro(5, 'Mapa de brechas por departamento (2023–2025)',
      'Identificar en qué departamentos la distancia entre lo urbano y lo rural es mayor o menor.',
      'Cada fila es un departamento: punto naranja = rural, punto azul = urbano, la línea es la brecha. Ordenado de menor a mayor brecha. Se muestran 24 de 25 departamentos (Callao no tiene muestra rural suficiente).')

# CELDA 8 – GRÁFICO 5: BRECHA POR DEPARTAMENTO (2023–2025)
# Compara urbano vs rural en secundaria completa o más (25+ años) para cada departamento.
# Solo incluye departamentos con al menos 300 casos en cada área; la línea es la brecha.
DEP = {1:'Amazonas',2:'Áncash',3:'Apurímac',4:'Arequipa',5:'Ayacucho',6:'Cajamarca',7:'Callao',8:'Cusco',9:'Huancavelica',10:'Huánuco',11:'Ica',12:'Junín',
       13:'La Libertad',14:'Lambayeque',15:'Lima',16:'Loreto',17:'Madre de Dios',18:'Moquegua',19:'Pasco',20:'Piura',21:'Puno',22:'San Martín',23:'Tacna',24:'Tumbes',25:'Ucayali'}
R = P[(P.anio >= 2023) & (P.edad >= 25)]
r = R.groupby(['depto','area']).apply(lambda g: pd.Series({'pct': wm(g.nivel.isin(SEC), g.peso), 'n': len(g)})).reset_index()
r = r.pivot(index='depto', columns='area', values=['pct','n'])
r = r[(r[('n','Rural')] >= 300) & (r[('n','Urbano')] >= 300)]       # solo departamentos con muestra suficiente en ambas áreas
d = pd.DataFrame({'dep': r.index.map(DEP), 'Urbano': r[('pct','Urbano')].values, 'Rural': r[('pct','Rural')].values})
d['brecha'] = d.Urbano - d.Rural; d = d.sort_values('brecha')
fig = go.Figure()
fig.add_trace(go.Scatter(x=d.Rural, y=d.dep, mode='markers', name='Rural', marker=dict(color=NAR, size=13)))
fig.add_trace(go.Scatter(x=d.Urbano, y=d.dep, mode='markers', name='Urbano', marker=dict(color=AZUL, size=13)))
for _, f in d.iterrows(): fig.add_shape(type='line', x0=f.Rural, x1=f.Urbano, y0=f.dep, y1=f.dep, line=dict(color='#bbb', width=3), layer='below')
fig.update_layout(title='<b>Secundaria completa o más (25+ años)</b> por departamento, 2023–2025<br><sup>La línea es la brecha; ordenado de menor a mayor brecha</sup>',
                  template='plotly_white', height=760, xaxis_title='% con secundaria completa o más', legend=dict(orientation='h', y=-0.08))
fig.write_html(os.path.join(OUT_CHARTS, 'grafico_05.html'), include_plotlyjs='cdn', full_html=True)
print('Departamentos incluidos:', len(d), '| Mayor brecha:', d.iloc[-1].dep, round(d.iloc[-1].brecha, 1), 'pp')

#@title Análisis gráfico 5 { display-mode: "form" }
analisis([
  'Mayor brecha: <b>La Libertad (40.2 pp)</b> y <b>Cajamarca (39.2 pp)</b>, seguidos de Moquegua (37.8) y Cusco (37.5).',
  'Menor brecha: <b>Madre de Dios (14.1 pp)</b>, Tumbes (16.1) e Ica (16.9).',
  'Rural más bajo: Cajamarca <b>16.0%</b>, La Libertad 18.6% y Huánuco 18.9% terminaron secundaria. Rural más alto: Ica 53.6%.',
  'Lima: urbano 74.3% vs rural 42.0% (brecha de 32.3 pp), así que la capital tampoco escapa a la brecha.',
  'Una brecha chica no siempre es buena señal: en Madre de Dios el urbano también es relativamente bajo (57.9%).'],
  'La brecha depende mucho del territorio: va de 14 pp a 40 pp, y el norte andino (Cajamarca, La Libertad) es donde más se siente.')

#@title Intro gráfico 6 { display-mode: "form" }
intro(6, 'Doble brecha: área y sexo',
      'Ver cómo se combinan la brecha urbano-rural y la brecha entre hombres y mujeres.',
      'Cuatro líneas: azul = urbano, naranja = rural; tono oscuro = hombres, tono claro = mujeres. Pasa el cursor para ver los valores de cada año.')

# CELDA 9 – GRÁFICO 6: DOBLE BRECHA (ÁREA Y SEXO)
# Evolución de secundaria completa o más (25+ años) para hombres y mujeres,
# en zona urbana y rural, para ver si el sexo amplía la brecha entre áreas.
S = P[P.edad >= 25]
s = S.groupby(['anio','area','sexo']).apply(lambda g: wm(g.nivel.isin(SEC), g.peso)).rename('pct').reset_index()
s['grupo'] = s.area + ' – ' + s.sexo
fig = px.line(s, x='anio', y='pct', color='grupo', markers=True, template='plotly_white', height=560,
              color_discrete_map={'Urbano – Hombre':'#1b4f93','Urbano – Mujer':'#6fa8ea','Rural – Hombre':'#b8431a','Rural – Mujer':'#f4a582'},
              labels={'pct':'% con secundaria completa o más','anio':'','grupo':''}, title='<b>Doble brecha</b>: área y sexo (25+ años)')
fig.update_traces(line_width=3, hovertemplate='%{y:.1f}%'); fig.update_layout(hovermode='x unified', xaxis=dict(dtick=2)); fig.write_html(os.path.join(OUT_CHARTS, 'grafico_06.html'), include_plotlyjs='cdn', full_html=True)

#@title Análisis gráfico 6 { display-mode: "form" }
analisis([
  'Mujer rural: <b>9.8% → 18.9%</b> con secundaria completa o más. Hombre rural: 18.6% → 31.8%.',
  'La mujer rural de 2025 (18.9%) recién iguala al hombre rural de 2004 (18.6%): <b>más de 20 años de retraso</b>.',
  'Brecha de género en el campo: 8.8 → <b>12.9 pp</b> (se amplió). En la ciudad se mantiene en ~11 pp (11.2 → 10.8).',
  'Distancia entre la mujer rural y el hombre urbano: <b>~53 pp</b> en 2004 y también ~53 pp en 2025 (62.9 − 9.8 vs 72.0 − 18.9).'],
  'La mujer rural acumula las dos desventajas: aunque todos mejoran, su distancia con el hombre urbano casi no cambió en 21 años.')

#@title Intro gráfico 7 { display-mode: "form" }
intro(7, 'Lengua materna y educación',
      'Ver si hablar una lengua originaria se asocia a menos educación, tanto en la ciudad como en el campo.',
      'Dos paneles: urbano (izquierda) y rural (derecha). Azul = castellano, morado = lengua originaria. En 2004 la pregunta de lengua solo está disponible en ~la mitad de los registros, así que ese año es menos fiable.')

# CELDA 10 – GRÁFICO 7: LENGUA MATERNA Y EDUCACIÓN
# Compara castellano vs lengua originaria en secundaria completa o más (25+ años),
# por separado en zona urbana y rural.
L = P[(P.edad >= 25) & P.lengua_g.isin(['Castellano','Lengua originaria'])]
l = L.groupby(['anio','area','lengua_g']).apply(lambda g: wm(g.nivel.isin(SEC), g.peso)).rename('pct').reset_index()
fig = px.line(l, x='anio', y='pct', color='lengua_g', facet_col='area', markers=True, template='plotly_white', height=520,
              color_discrete_map={'Castellano':'#2a78d6','Lengua originaria':'#7a3db8'}, category_orders={'area':['Urbano','Rural']},
              labels={'pct':'% con secundaria completa o más','anio':'','lengua_g':''}, title='<b>Lengua materna y educación</b> (25+ años)')
fig.update_traces(line_width=3); fig.update_layout(hovermode='x unified'); fig.write_html(os.path.join(OUT_CHARTS, 'grafico_07.html'), include_plotlyjs='cdn', full_html=True)

#@title Análisis gráfico 7 { display-mode: "form" }
analisis([
  'Rural: hablantes de lengua originaria <b>9.3% → 20.3%</b> con secundaria completa o más; hablantes de castellano 16.9% → 29.1%.',
  'Urbano: lengua originaria <b>26.4% → 39.9%</b>; castellano 63.3% → 71.5%. La distancia por lengua en la ciudad bajó de 36.9 a 31.6 pp.',
  'En el campo la distancia por lengua se mantuvo en ~8–9 pp (7.6 → 8.8), así que no se está cerrando.',
  'Un hablante de lengua originaria en el campo (20.3%) está a unos <b>51 pp</b> de un hablante de castellano en la ciudad (71.5%).'],
  'La lengua originaria suma desventaja encima de vivir en el campo, y esa distancia es la que más cuesta cerrar.')

#@title Intro gráfico 8 { display-mode: "form" }
intro(8, 'Burbujas por departamento: educación vs ruralidad, 2004–2025',
      'Ver, departamento por departamento, cómo se relacionan la ruralidad y la educación, y cómo se movieron en 21 años.',
      'Cada burbuja es un departamento: eje X = % de población rural, eje Y = % de adultos (25+) con secundaria completa o más, tamaño = población, color = región (Costa, Sierra, Selva). Pulsa ▶ para ver 2004 → 2025.')

# CELDA 11 – GRÁFICO 8: BURBUJAS ESTILO ROSLING POR DEPARTAMENTO (ANIMADO)
# Cada burbuja es un departamento. X = % de población rural,
# Y = % de adultos (25+) con secundaria completa o más, tamaño = población.
# Pulsa ▶ para ver 2004 → 2025.
REG = {**{d: 'Costa' for d in [7, 11, 13, 14, 15, 18, 20, 23, 24]},
       **{d: 'Sierra' for d in [2, 3, 4, 5, 6, 8, 9, 10, 12, 19, 21]},
       **{d: 'Selva' for d in [1, 16, 17, 22, 25]}}

def resumen_depto(g):
    a25 = g[g.edad >= 25]
    return pd.Series({
        'pct_rural': wm(g.area.eq('Rural'), g.peso),
        'pct_secundaria': wm(a25.nivel.isin(SEC), a25.peso),
        'poblacion': g.peso.sum(),
    })

B = (P.groupby(['anio', 'depto'])[['edad', 'area', 'nivel', 'peso']]
       .apply(resumen_depto).reset_index())
B['departamento'] = B.depto.map(DEP)
B['region'] = B.depto.map(REG)
B.to_csv('burbujas_departamento.csv', index=False)

fig = px.scatter(B, x='pct_rural', y='pct_secundaria', size='poblacion', color='region',
                 animation_frame='anio', animation_group='departamento', hover_name='departamento',
                 size_max=60, range_x=[-5, 85], range_y=[0, 90], template='plotly_white', height=650,
                 color_discrete_map={'Costa': '#2a78d6', 'Sierra': '#eb6834', 'Selva': '#2e9d6a'},
                 labels={'pct_rural': '% de población rural', 'pct_secundaria': '% de adultos (25+) con secundaria completa o más',
                         'region': 'Región predominante', 'poblacion': 'Población', 'anio': 'Año'},
                 title='<b>20 años de educación en el Perú, departamento por departamento</b><br>'
                       '<sup>Al estilo de Hans Rosling · ENAHO 2004–2025 · tamaño = población</sup>')
fig.update_traces(marker=dict(line=dict(width=1, color='white')))
fig.write_html(os.path.join(OUT_CHARTS, 'grafico_08.html'), include_plotlyjs='cdn', full_html=True)

# Departamentos que más avanzaron (para el artículo)
cmp_ = B[B.anio.isin([2004, 2025])].pivot(index='departamento', columns='anio', values='pct_secundaria')
cmp_['avance_pp'] = cmp_[2025] - cmp_[2004]
display(cmp_.sort_values('avance_pp', ascending=False).round(1))

#@title Análisis gráfico 8 { display-mode: "form" }
analisis([
  'Relación fuerte: mientras más rural es un departamento, menos adultos terminaron la secundaria (correlación de <b>−0.93</b> en 2004 y <b>−0.87</b> en 2025).',
  '2004: Huancavelica era el más rural (<b>82.2%</b> de población rural) y solo <b>15.0%</b> de sus adultos tenía secundaria; Lima y Callao llegaban a 63.2%.',
  '2025: Lima (<b>74.3%</b>) y Callao (74.2%) siguen primeros; Huánuco (<b>29.6%</b>) y Cajamarca (31.7%) quedan últimos.',
  'Casi todas las burbujas se mueven hacia arriba y a la izquierda: se vuelven menos rurales y más educadas. Amazonas bajó de 68.3% a 41.0% de población rural; Puno, de 59.1% a 35.1%.',
  'Mayor avance en secundaria: Moquegua (<b>+22.0 pp</b>), Huancavelica (+19.2 pp) y Ayacucho (+17.0 pp). Menor avance: Loreto (+8.1 pp), Huánuco (+9.0 pp) y Apurímac (+10.6 pp).',
  'Por región (promedio ponderado por población): Costa 54.9% → 67.7%; Sierra 31.8% → 46.2%; Selva 31.4% → 42.8%.',
  'El último departamento de la lista pasó de 15.0% a 29.6% y el primero, de 63.2% a 74.3%: la distancia entre extremos bajó de 48.2 a 44.7 pp.'],
  'Donde hay más ruralidad hay menos educación, y todos los departamentos mejoran; pero los más rurales (Huánuco, Cajamarca, Huancavelica) siguen más de 40 pp por debajo de Lima.')

#@title Intro modelo { display-mode: "form" }
intro('MODELO', 'Árbol de decisión: ¿se puede predecir quién terminó la secundaria?',
      'Probar si el contexto de una persona (área, sexo, lengua materna, departamento y edad) alcanza para predecir si terminó la secundaria, y medir qué tan bien lo hace.',
      'Datos 2023–2025, personas de 25+ años, 70% para entrenar y 30% para probar. La tabla compara el modelo con una línea base ("siempre decir que terminó"): si no la supera, el modelo no aporta nada.')

# CELDA 13 – MODELO: ÁRBOL DE DECISIÓN
# Pregunta: ¿se puede predecir si un adulto (25+) terminó la secundaria
# solo con su contexto (área, sexo, lengua materna, departamento, edad)?
# Datos: 2023–2025. 70 % entrenamiento / 30 % prueba. Ponderado con el factor de expansión.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, balanced_accuracy_score

M = P[(P.anio >= 2023) & (P.edad >= 25) & P.peso.notna()].copy()
M['y'] = M.nivel.isin(SEC).astype(int)                       # 1 = secundaria completa o más
M['departamento'] = M.depto.map(DEP)
X = pd.get_dummies(M[['area', 'sexo', 'lengua_g', 'departamento']], drop_first=False).astype(int)
X['edad'] = M.edad.values
y, w = M.y.values, M.peso.values

X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(X, y, w, test_size=0.3, random_state=42, stratify=y)
modelo = DecisionTreeClassifier(max_depth=6, min_samples_leaf=200, random_state=42)
modelo.fit(X_tr, y_tr, sample_weight=w_tr)
pred = modelo.predict(X_te)

# Línea base: predecir siempre la clase más frecuente (para saber si el modelo aporta algo)
base_clase = int(np.average(y_tr, weights=w_tr) >= 0.5)
base_acc = accuracy_score(y_te, np.full_like(y_te, base_clase), sample_weight=w_te)

met = {
    'Exactitud (accuracy)': accuracy_score(y_te, pred, sample_weight=w_te),
    'Exactitud balanceada': balanced_accuracy_score(y_te, pred, sample_weight=w_te),
    'Recall: detecta a quienes NO terminaron': recall_score(y_te, pred, pos_label=0, sample_weight=w_te),
    'Precisión al marcar "no terminó"': precision_score(y_te, pred, pos_label=0, sample_weight=w_te),
    'Línea base (siempre la clase mayoritaria)': base_acc,
}
display(pd.Series(met).mul(100).round(1).rename('%').to_frame())
print(f'Registros: {len(M):,} | entrenamiento: {len(X_tr):,} | prueba: {len(X_te):,}')

#@title Análisis modelo { display-mode: "form" }
analisis([
  'Exactitud: <b>73.6%</b> frente a <b>58.2%</b> de la línea base (decir siempre "terminó"). El modelo mejora en unos 15 puntos.',
  'Exactitud balanceada: <b>72.6%</b>. Acierta de forma parecida en los dos grupos y no depende de que haya más personas que terminaron.',
  'Recall <b>66.9%</b>: detecta a 2 de cada 3 personas que no terminaron la secundaria. Precisión <b>69.0%</b>: cuando dice "no terminó", acierta cerca de 7 de cada 10 veces.',
  'Se usaron 198,335 registros: 138,834 para entrenar y 59,501 para probar.',
  'Ojo: el modelo muestra asociaciones, no causas.'],
  'Solo con el contexto (dónde vive, qué lengua habla, su sexo y su edad) se predice bastante bien quién terminó la secundaria, pero no del todo: se equivoca en casi 1 de cada 4 casos.')

#@title Intro gráfico 9 { display-mode: "form" }
intro(9, 'Matriz de confusión: ¿en qué se equivoca el modelo?',
      'Ver en qué casos acierta y en cuáles se equivoca el árbol de decisión, según lo que la persona realmente es.',
      'Cada fila es la realidad (terminó o no la secundaria) y cada columna lo que el modelo predijo. Los % de cada fila suman 100. Los aciertos son los cuadros donde "Real" y "Predijo" coinciden.')

# CELDA 14 – GRÁFICO 9: MATRIZ DE CONFUSIÓN (amigable)
# Porcentaje de personas reales de cada grupo que el modelo clasificó bien o mal.
cm = confusion_matrix(y_te, pred, sample_weight=w_te)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
etq = ['No terminó secundaria', 'Terminó secundaria']
texto = [[f'{cm_pct[i, j]:.0f}%' for j in range(2)] for i in range(2)]
fig = go.Figure(go.Heatmap(z=cm_pct, x=['Predijo: no terminó', 'Predijo: terminó'], y=['Real: no terminó', 'Real: terminó'],
                           text=texto, texttemplate='<b>%{text}</b>', textfont=dict(size=26),
                           colorscale=[[0, '#f2f7ff'], [1, '#2a78d6']], showscale=False,
                           hovertemplate='%{y} · %{x}: %{z:.1f}%<extra></extra>'))
fig.update_layout(title=f'<b>¿Acierta el modelo?</b> Matriz de confusión (prueba, 2023–2025)<br>'
                        f'<sup>Exactitud {met["Exactitud (accuracy)"]*100:.0f}% · línea base {base_acc*100:.0f}% · cada fila suma 100%</sup>',
                  template='plotly_white', height=480, width=640, yaxis=dict(autorange='reversed'))
fig.write_html(os.path.join(OUT_CHARTS, 'grafico_09.html'), include_plotlyjs='cdn', full_html=True)

#@title Análisis gráfico 9 { display-mode: "form" }
analisis([
  'De las personas que <b>no terminaron</b> la secundaria, el modelo acierta con <b>67%</b>; el otro <b>33%</b> lo clasifica como si hubiera terminado.',
  'De las personas que <b>sí terminaron</b>, acierta con <b>78%</b>; se equivoca con el 22%.',
  'Es mejor detectando a quienes terminaron que a quienes no: 1 de cada 3 personas que no terminó pasa desapercibida.'],
  'El contexto da una buena pista, pero no basta: dos personas con el mismo perfil (misma área, lengua, sexo y edad) pueden tener trayectorias distintas.')

#@title Intro gráfico 10 { display-mode: "form" }
intro(10, '¿Qué pesa más? Importancia de las variables',
      'Identificar qué características de una persona pesan más al predecir si terminó la secundaria.',
      'Cada barra suma la importancia de una variable dentro del árbol (el total es 100%). Barra más larga = esa variable pesa más en la predicción.')

# CELDA 15 – GRÁFICO 10: ¿QUÉ PESA MÁS? (importancia de variables)
# Agrupa las columnas dummy en su variable original (área, sexo, lengua, departamento, edad).
imp = pd.Series(modelo.feature_importances_, index=X.columns)
grupo = lambda c: ('Área (urbano/rural)' if c.startswith('area_') else 'Sexo' if c.startswith('sexo_')
                   else 'Lengua materna' if c.startswith('lengua_g_') else 'Departamento' if c.startswith('departamento_') else 'Edad')
imp_g = imp.groupby(imp.index.map(grupo)).sum().sort_values() * 100
fig = go.Figure(go.Bar(x=imp_g.values, y=imp_g.index, orientation='h', marker_color='#2a78d6',
                       text=[f'{v:.0f}%' for v in imp_g.values], textposition='outside'))
fig.update_layout(title='<b>¿Qué pesa más para predecir si alguien terminó la secundaria?</b><br><sup>Importancia de cada variable en el árbol de decisión</sup>',
                  template='plotly_white', height=420, xaxis=dict(title='% de importancia', range=[0, imp_g.max() * 1.2]))
fig.write_html(os.path.join(OUT_CHARTS, 'grafico_10.html'), include_plotlyjs='cdn', full_html=True)
display(imp_g.round(1).rename('importancia_%').to_frame())

#@title Análisis gráfico 10 { display-mode: "form" }
analisis([
  'Área (urbano/rural) y edad pesan casi igual: <b>37.1%</b> y <b>37.0%</b>. Juntas explican cerca de tres cuartos de las decisiones del árbol.',
  'Lengua materna: <b>13.2%</b>. Departamento: <b>8.7%</b>. Sexo: <b>4.0%</b>.',
  'La edad refleja el relevo generacional que vimos en el gráfico 4: las personas más jóvenes llegan más lejos.',
  'Ojo: importancia no es causalidad, y variables como área y departamento se superponen (los departamentos más rurales también son los de menor nivel educativo).'],
  'Vivir en el campo pesa tanto como la edad al predecir quién terminó la secundaria. Ser mujer u hombre pesa poco en este modelo.')

# CELDA 12 – RESUMEN DE HALLAZGOS
# Muestra una tabla resumen con el cambio 2004 → 2025 en urbano, rural y brecha.
# Verde = la brecha se achicó; rojo = la brecha se amplió. Útil para el texto del post.
from IPython.display import HTML, display

def v(ind, y, col): return IND_W[(IND_W.indicador == ind) & (IND_W.anio == y)][col].iloc[0]

filas = ''
for i, ind in enumerate(INDS):
    es_anios = 'Años' in ind
    f = (lambda x: f'{x:.1f} años') if es_anios else (lambda x: f'{x:.1f}%')
    u0, u1, r0, r1 = v(ind,2004,'Urbano'), v(ind,2025,'Urbano'), v(ind,2004,'Rural'), v(ind,2025,'Rural')
    b0, b1 = abs(v(ind,2004,'brecha')), abs(v(ind,2025,'brecha'))
    achico = b1 < b0
    color, bg = ('#1a7f4b', '#e8f6ee') if achico else ('#c0392b', '#fdecea')
    flecha, txt = ('▼', 'Se achicó') if achico else ('▲', 'Se amplió')
    ub = 'años' if es_anios else 'pp'
    fondo = '#f7f9fc' if i % 2 else '#ffffff'
    filas += f'''<tr style="background:{fondo}">
      <td style="padding:12px 14px;font-weight:600;color:#1f2d3d">{ind}</td>
      <td style="padding:12px 14px;text-align:center;color:#2a78d6">{f(u0)} <span style="color:#999">→</span> <b>{f(u1)}</b></td>
      <td style="padding:12px 14px;text-align:center;color:#eb6834">{f(r0)} <span style="color:#999">→</span> <b>{f(r1)}</b></td>
      <td style="padding:12px 14px;text-align:center;color:#1f2d3d">{b0:.1f} <span style="color:#999">→</span> <b>{b1:.1f}</b> {ub}</td>
      <td style="padding:8px 14px;text-align:center"><span style="background:{bg};color:{color};padding:5px 12px;border-radius:14px;font-weight:700;font-size:13px">{flecha} {txt} ({b1-b0:+.1f})</span></td>
    </tr>'''

display(HTML(f'''
<div style="font-family:Segoe UI,Roboto,Arial,sans-serif;max-width:1000px">
  <div style="font-size:22px;font-weight:700;color:#1f2d3d;margin:6px 0 2px">Brecha educativa urbano vs rural: 2004 → 2025</div>
  <div style="font-size:13px;color:#6b7785;margin-bottom:12px">ENAHO (INEI) · Módulo de Educación · ponderado con factor de expansión</div>
  <table style="border-collapse:collapse;width:100%;border-radius:12px;overflow:hidden;box-shadow:0 2px 10px rgba(0,0,0,.12);font-size:14px">
    <thead><tr style="background:#1f3a5f;color:white">
      <th style="padding:12px 14px;text-align:left">Indicador</th>
      <th style="padding:12px 14px">Urbano</th><th style="padding:12px 14px">Rural</th>
      <th style="padding:12px 14px">Brecha (absoluta)</th><th style="padding:12px 14px">¿Qué pasó?</th>
    </tr></thead>
    <tbody>{filas}</tbody>
  </table>
  <div style="font-size:12px;color:#8a95a3;margin-top:10px">Brecha = diferencia absoluta entre urbano y rural (puntos porcentuales; años en "Años promedio de estudio"). 2020: encuesta telefónica por COVID.</div>
</div>'''))

# Al terminar, los 10 gráficos originales estarán en:
print('Gráficos originales exportados a:', OUT_CHARTS)
print('Archivos:', sorted(os.listdir(OUT_CHARTS)))
